In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("init_load_flag", "")
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

In [0]:
df = spark.read.table("databricks_cata.silver.regions_silver")

In [0]:
df.display()

In [0]:
df = df.dropDuplicates(subset=["region_id"])

In [0]:
# Region is a static dimension - no SCD2 needed
# No row_hash required since we are not tracking history
# Simple SCD1 overwrite on match is sufficient

if init_load_flag == 0:
    df_old = spark.sql('''select DimRegionKey, region_id, create_date, update_date
                       from databricks_cata.gold.DimRegion''')
else:
    df_old = spark.sql('''select 0 DimRegionKey, 0 region_id, 0 create_date, 0 update_date
                       from databricks_cata.silver.regions_silver
                       where 1=2''')

In [0]:
df_old.display()

In [0]:
df_old = df_old\
    .withColumnRenamed("DimRegionKey", "old_DimRegionKey")\
    .withColumnRenamed("region_id", "old_region_id")\
    .withColumnRenamed("create_date", "old_create_date")\
    .withColumnRenamed("update_date", "old_update_date")

In [0]:
df_old.display()

In [0]:
df_join = df.join(df_old, df.region_id == df_old.old_region_id, "left")

In [0]:
# New regions - no match in gold
df_new = df_join.filter(df_join.old_DimRegionKey.isNull())
df_new.display()

In [0]:
# Existing regions - already in gold (SCD1 - will update in place via merge)
df_existing = df_join.filter(df_join.old_DimRegionKey.isNotNull())
print("existing regions count:", df_existing.count())

In [0]:
df_new = df_new.select(
    df_new.region_id,
    df_new.region
)

df_new = df_new\
    .withColumn("create_date", current_timestamp())\
    .withColumn("update_date", current_timestamp())

df_new.display()

In [0]:
w = Window.partitionBy(lit(1)).orderBy("region_id")
df_new = df_new.withColumn("DimRegionKey", row_number().over(w))

In [0]:
if init_load_flag == 1:
    max_surrogate_key = 0
else:
    df_maxsur = spark.sql("select max(DimRegionKey) as max_surrogate_key from databricks_cata.gold.DimRegion")
    max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']
    if max_surrogate_key is None:
        max_surrogate_key = 0

In [0]:
df_new = df_new.withColumn("DimRegionKey", lit(max_surrogate_key) + col("DimRegionKey"))
df_new.limit(5).display()

In [0]:
if spark.catalog.tableExists("databricks_cata.gold.DimRegion"):
    dlt_obj = DeltaTable.forPath(spark, "abfss://gold@customersprojectete.dfs.core.windows.net/DimRegion")

    dlt_obj.alias("tgt").merge(
        df_new.alias("src"),
        "tgt.region_id = src.region_id"
    ).whenMatchedUpdate(set={
        "region": col("src.region"),
        "update_date": current_timestamp()
    }).whenNotMatchedInsertAll().execute()

else:
    df_new.write.mode("overwrite")\
        .format("delta")\
        .option("path", "abfss://gold@customersprojectete.dfs.core.windows.net/DimRegion")\
        .saveAsTable("databricks_cata.gold.DimRegion")

In [0]:
%sql
select * from databricks_cata.gold.DimRegion order by DimRegionKey